<a href="https://colab.research.google.com/github/joduhwan1/test-repo/blob/master/llama_part1_Data_Preparation_hkcode_youtube_ipynb%EC%9D%98_%EC%82%AC%EB%B3%B8%EC%9D%98_%EC%82%AC%EB%B3%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Google Drive Connection

In [ ]:
from google.colab import drive
drive.mount("/content/gdrive")

Mounted at /content/gdrive


★ setting point: Set the folder path where your data is stored.

example: "/content/gdrive/MyDrive/Colab Notebooks/llama/01. data preparation/dataset"


In [ ]:
dataPath = "/content/gdrive/MyDrive/Colab Notebooks/llama/01. data preparation/dataset"

### Install Library

In [ ]:
# datasets: loads and manages data from hugging-face's community.
# jsonlines: handles json format data
!pip install datasets==2.16.1 jsonlines==4.0.0

INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.1/507.1 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.4/166.4 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 18.6 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.6.1
    Uninstalling fsspec-2024.6.1:
      Successfully uninstalled fsspec-2024.6.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.3.1+cu121 requires nvidia-cublas-cu12==12.1.3.1; platform_system == "Linux" and platform_machine == "x

### Login to access huggign face

In [ ]:
# How to check HF access token: hugging face -> select account at top right -> settings -> access token
import huggingface_hub
huggingface_hub.login()

### Define installed libraries

In [ ]:
import pandas as pd
import json
import jsonlines
from datasets import Dataset
import os

# ★ Hugging face 없이 csv 바로 튜닝데이터로 변환 ★ HF 호환

In [ ]:
# Define paths and filenames
fileName = "indata_kor"
dataset_Path = "/content/gdrive/MyDrive/Colab Notebooks/llama/01. data preparation/dataset"
csv_file_path = os.path.join(dataset_Path, f"{fileName}.csv")

def csv_to_dataset(csv_file_path):
    try:
        df = pd.read_csv(csv_file_path, encoding="ms949")
    except Exception as e:
        df = pd.read_csv(csv_file_path, encoding="utf-8")

    # Initialize an empty list to store formatted strings
    indataset = []

    # Iterate over the range of the DataFrame length
    for index in range(len(df)):
        row = df.iloc[index]
        formatted_string = f'<s>[INST] {row["inputs"]} [/INST] {row["response"]} </s>'
        # formatted_string = f"""<|begin_of_text|><|start_header_id|>user<|end_header_id|>{""}<|eot_id|><|start_header_id|>assistant<|end_header_id|>{row["inputs"]}<|eot_id|><|start_header_id|>user<|end_header_id|>{row["response"]}<|eot_id|>"""

        formatted_string = formatted_string.replace("\n", "")
        indataset.append(formatted_string.lstrip())

    # Create and return the dataset
    dataset = Dataset.from_dict({"text": indataset})
    return dataset

# Load the dataset directly from the CSV file
dataset = csv_to_dataset(csv_file_path)

# Display the first 28 entries of the dataset
print('dataset')
print(dataset['text'][:28])
dataset[28]
# 필요시 허깅페이스에 업로드
dataset.push_to_hub("hyokwan/hkcode_korea")

# 1. Import training data

★ setting point: Set your own training data file name.
example: indata_kor.csv


In [ ]:
# Setting data paths
datasetName = "indata_kor.csv"

In [ ]:
def csv_to_json(csv_file_path, json_file_path):
    df = pd.read_csv(csv_file_path, encoding="ms949")

    # save as json file
    with open(json_file_path, 'w', encoding='utf-8') as json_file:
        # Convert each row to JSON and write directly to file
        for index, row in df.iterrows():
            data = {'inputs': row['inputs'], 'response': row['response']}
            json.dump(data, json_file, ensure_ascii=False)
            json_file.write('\n')

# Set CSV file path and JSON file path
csv_file_path = os.path.join( dataPath, datasetName)
jsonName = datasetName.split(".")[0]+".jsonl"
json_file_path = os.path.join( dataPath, jsonName)

csv_to_json(csv_file_path, json_file_path)

# 2. Convert to finetuning format

In [ ]:
indataset = []
with jsonlines.open(json_file_path) as f:
    for line in f.iter():
      indataset.append(f'<s>[INST] {line["inputs"]} [/INST] {line["response"]} </s>')
    #   indataset.append(f'<s>### Instruction: \n{line["inputs"]} \n\n### Response: \n{line["response"]}</s>')

print('dataset')
print(indataset[:5])

# Create and save datasets
indataset = Dataset.from_dict({"text": indataset})
indataset.save_to_disk(dataPath)

# Check Dataset info
print('Check Dataset info')
print(indataset)

dataset
['<s>[INST] 유튜브 채널 hkcode에서는 무엇을 가르치나요? [/INST] 초보자 대상으로 빅데이터, 인공지능과 관련된 컨텐츠를 가르치고 있습니다. </s>', '<s>[INST] 유튜브 채널 hkcode는 누가 운영하나요? [/INST] 한국폴리텍대학 스마트금융과 김효관 교수가 운영합니다. </s>', '<s>[INST] 스마트금융과는 무엇을 가르치나요? [/INST] 스마트금융과는 빅데이터, 인공지능, 웹개발 및 블록체인을 가르치고 있습니다. </s>', '<s>[INST] 스마트금융과 등록비용은 얼마인가요? [/INST] 등록비용은 국비지원 과정으로 무료 입니다. </s>', '<s>[INST] 스마트금융과는 1년에 몇 명을 선발하나요? [/INST] 1년에 한반을 운영하고 있고 최대 27명을 선발합니다. </s>']


Saving the dataset (0/1 shards):   0%|          | 0/32 [00:00<?, ? examples/s]

Check Dataset info
Dataset({
    features: ['text'],
    num_rows: 32
})


# 3. Upload data to huggingface (★ Change Point!)

In [ ]:
indataset.push_to_hub("hyokwan/hkcode_korea")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

CommitInfo(commit_url='https://huggingface.co/datasets/hyokwan/hkcode_korea/commit/e33493f2e26412c0ae880034067d6cd57c1abc17', commit_message='Upload dataset', commit_description='', oid='e33493f2e26412c0ae880034067d6cd57c1abc17', pr_url=None, pr_revision=None, pr_num=None)

In [ ]:
indataset[28]

{'text': '<s>[INST] 한국폴리텍대학 스마트금융과의 최종 포트폴리오는 어떤건가요? [/INST] 유튜브 채널에서 스마트금융과를 검색하시면 한국폴리텍대학 스마트금융과 포트폴리오 발표라는 재생목록을 통해 확인할 수 있습니다. </s>'}

# HuggingFace 없이 csv 파일만 json  변환 저장 후 활용 시

In [ ]:
!pip install datasets==2.16.1 jsonlines==4.0.0
import jsonlines

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.1/507.1 kB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 8.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 7.6 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 8.2 MB/s eta 0:00:00


In [ ]:
import os

In [ ]:
# 본인의 파일명
fileName = "indata_kor"
dataset_Path = "/content/gdrive/MyDrive/Colab Notebooks/llama/01. data preparation/dataset"
csv_file_path = os.path.join(dataset_Path, f"{fileName}.csv")
json_file_path = os.path.join(dataset_Path, f"{fileName}.jsonl")

In [ ]:
def csv_to_json(csv_file_path, json_file_path):
    df = pd.read_csv(csv_file_path, encoding="ms949")

    # save as json file
    with open(json_file_path, 'w', encoding='utf-8') as json_file:
        # Convert each row to JSON and write directly to file
        for index, row in df.iterrows():
            data = {'inputs': row['inputs'], 'response': row['response']}
            json.dump(data, json_file, ensure_ascii=False)
            json_file.write('\n')

# Set CSV file path and JSON file path
csv_file_path = os.path.join( dataPath, datasetName)
jsonName = datasetName.split(".")[0]+".jsonl"
json_file_path = os.path.join( dataPath, jsonName)

csv_to_json(csv_file_path, json_file_path)

In [ ]:
indataset = []
with jsonlines.open(json_file_path) as f:
    for line in f.iter():
      indataset.append(f'<s>[INST] {line["inputs"]} [/INST] {line["response"]} </s>')
    #   indataset.append(f'<s>### Instruction: \n{line["inputs"]} \n\n### Response: \n{line["response"]}</s>')

print('dataset')
print(indataset[:5])

# Create and save datasets
dataset = Dataset.from_dict({"text": indataset})

dataset
['<s>[INST] 유튜브 채널 hkcode에서는 무엇을 가르치나요? [/INST] 초보자 대상으로 빅데이터, 인공지능과 관련된 컨텐츠를 가르치고 있습니다. </s>', '<s>[INST] 유튜브 채널 hkcode는 누가 운영하나요? [/INST] 한국폴리텍대학 스마트금융과 김효관 교수가 운영합니다. </s>', '<s>[INST] 스마트금융과는 무엇을 가르치나요? [/INST] 스마트금융과는 빅데이터, 인공지능, 웹개발 및 블록체인을 가르치고 있습니다. </s>', '<s>[INST] 스마트금융과 등록비용은 얼마인가요? [/INST] 등록비용은 국비지원 과정으로 무료 입니다. </s>', '<s>[INST] 스마트금융과는 1년에 몇 명을 선발하나요? [/INST] 1년에 한반을 운영하고 있고 최대 27명을 선발합니다. </s>']


### PART2 파인튜닝용 데이터셋!

In [ ]:
dataset[28]

{'text': '<s>[INST] 한국폴리텍대학 스마트금융과의 최종 포트폴리오는 어떤건가요? [/INST] 유튜브 채널에서 스마트금융과를 검색하시면 한국폴리텍대학 스마트금융과 포트폴리오 발표라는 재생목록을 통해 확인할 수 있습니다. </s>'}